In [94]:
pip install refinitiv-data pandas

Note: you may need to restart the kernel to use updated packages.


In [95]:
import pandas as pd
import refinitiv.data as rd
import kit1 as kit

In [96]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x129495f40 {name='workspace'}>

In [97]:
# Hàm này mô phỏng lại đúng cái bạn cần
def load_config(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [98]:
# Sử dụng
CFG = load_config("config.json")


In [99]:
# 1. Bọc dấu ngoặc kép (") cho từng mã thị trường, sau đó nối bằng dấu phẩy
markets_str = ",".join([f'"{m}"' for m in CFG["markets"]])

# 2. Đưa vào query (Lưu ý: Không bọc thêm ngoặc kép quanh {markets_str} ở bước này nữa)
screener_expr = f'SCREEN(U(IN(TR.ExchangeMarketIdCode,{markets_str})), TR.TRESGScore>={CFG["min_esg_score"]})'

# Print ra để kiểm tra nhanh (tùy chọn)
# print(screener_expr) 
# Kết quả kỳ vọng: SCREEN(U(IN(TR.ExchangeMarketIdCode,"XPAR","XAMS",...)), TR.TRESGScore>=50)

In [100]:
df = rd.get_history(
    universe="GOOG.O",
    fields=["BID", "ASK"],
    interval="tick",
    count=5
)
print(df)

2026-02-25 19:44:19,941 - HTTP Request: GET http://localhost:9005/api/rdp/data/historical-pricing/v1/views/events/GOOG.O?count=5&fields=BID%2CASK%2CDATE_TIME "HTTP/1.1 200 OK"


GOOG.O                      BID     ASK
Timestamp                              
2026-02-25 18:29:17.739  310.36  310.38
2026-02-25 18:29:17.740  310.36  310.38
2026-02-25 18:29:17.741  310.35  310.38
2026-02-25 18:29:18.114  310.36  310.38
2026-02-25 18:29:18.178  310.36  310.38


In [101]:
print(CFG["markets"])

['XPAR', 'XAMS', 'XBRU', 'XMAD', 'XNYS', 'XNAS']


In [102]:
mic_codes = ", ".join([f"'{m}'" for m in CFG.get("markets", list(CFG["markets"]))])
screener_query = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))), "
    f"IN(TR.ExchangeMarketIdCode, {mic_codes}), "
    "TR.TRESGScore > 50"
    ")"
)


In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [104]:
# Gọi API lấy dữ liệu
df_screener = rd.get_data(
    universe=screener_query,
    fields=[
        "TR.CommonName",
        "TR.ISIN",
        "TR.ExchangeName",
        "TR.ExchangeMarketIdCode",
        "TR.Currency",
        "TR.TRESGScore",
        "TR.TRESGGrade",
        "TR.CompanyMarketCap",
        "TR.PriceClose",
    ]
    # Đã xóa dòng use_field_names_in_headers để không bị báo FutureWarning nữa
)

# In thử 5 dòng đầu
display(df_screener.head())

2026-02-25 19:44:28,168 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
/opt/anaconda3/envs/final_project/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


,Instrument,Company Common Name,ISIN,Exchange Name,Exchange Market Identifier Code,ESG Score,Company Market Cap,Price Close
0,FN.N,Fabrinet,KYG3323L1005,New York Stock Exchange,XNYS,58.673081,20989447166.790001,585.87
1,TGLS.N,Tecnoglass Inc,KYG872641009,New York Stock Exchange,XNYS,70.471181,2326609522.16,49.96
2,HPP.N,Hudson Pacific Properties Inc,US4440974065,New York Stock Exchange,XNYS,65.939388,340485315.96,6.28
3,AMA.MC,Amadeus IT Group SA,ES0109067019,BME - BOLSAS Y MERCADOS ESPANOLES,XMAD,89.508399,22200600822.400002,49.28
4,ZWS.N,Zurn Elkay Water Solutions Corp,US98983L1089,New York Stock Exchange,XNYS,69.358455,8441645464.14,50.61


In [ ]:
CFG["history_start"] = "2025-01-06"

In [106]:
import pandas as pd
import refinitiv.data as rd
from datetime import datetime, timedelta
import os
from tqdm.auto import tqdm

FILE_PATH = "quant_master_db.csv"

# Giả sử cậu đã có current_universe từ list ESG hợp lệ
# current_universe = df_screener['Instrument'].tolist() 
end_date = datetime.now().strftime('%Y-%m-%d')

print(f"🔄 BẮT ĐẦU CẬP NHẬT DATABASE CHO {len(current_universe)} MÃ...")

# ==========================================
# BƯỚC 1: XỬ LÝ DATABASE CŨ & LOGIC NGÀY KÉO ĐÈ (OVERLAP)
# ==========================================
if os.path.exists(FILE_PATH):
    # Đọc file thấp cấp dạng string trước để tránh lỗi dính dòng rác
    df_old = pd.read_csv(FILE_PATH, dtype=str)
    
    # Ép kiểu cột Date cực mạnh, dòng nào rác/chữ/lỗi sẽ biến thành NaT (Not a Time)
    df_old['Date'] = pd.to_datetime(df_old['Date'], errors='coerce')
    
    # Lọc bỏ các dòng NaT (rác) và ép kiểu cột PriceClose về số thực
    df_old = df_old.dropna(subset=['Date'])
    df_old['PriceClose'] = pd.to_numeric(df_old['PriceClose'], errors='coerce')
    
    # Dọn dẹp: Chỉ giữ lại những mã đang có trong current_universe để file không bị phình to
    df_old = df_old[df_old['RIC'].isin(current_universe)]
    
    last_date = df_old['Date'].max()
    
    # 🎯 CHIẾN THUẬT KÉO ĐÈ: Lùi lại 5 ngày để bao trọn cuối tuần/nghỉ lễ
    # Không dùng +1 ngày nữa để tránh API Refinitiv trả về lỗi rỗng
    start_date = (last_date - timedelta(days=5)).strftime('%Y-%m-%d')
    
    print(f"📅 DB cũ lưu đến ngày: {last_date.strftime('%Y-%m-%d')}")
    print(f"🛡️ Kéo đè an toàn từ: {start_date} đến {end_date} (Sẽ tự động xóa trùng lặp)")

else:
    print("⚠️ Không tìm thấy DB, khởi tạo mới hoàn toàn (Lấy 252 ngày qua)...")
    df_old = pd.DataFrame()
    start_date = (datetime.now() - timedelta(days=252)).strftime('%Y-%m-%d')

# ==========================================
# BƯỚC 2: KÉO DỮ LIỆU TỪ REFINITIV
# ==========================================
new_data_list = []

# Chặn trường hợp start_date vượt quá end_date
if pd.to_datetime(start_date) >= pd.to_datetime(end_date):
    print("✅ Dữ liệu đã là mới nhất của ngày hôm nay, không cần kéo thêm API!")
else:
    print(f"🚀 Đang tải dữ liệu...")
    for ric in tqdm(current_universe):
        try:
            df_temp = rd.get_history(
                universe=ric,
                fields=['TR.PriceClose'], 
                start=start_date,
                end=end_date,
                interval="1D"
            )
            
            # Kiểm tra an toàn: Có dữ liệu và ít nhất 1 dòng
            if df_temp is not None and not df_temp.empty:
                df_temp = df_temp.reset_index()
                
                df_clean = pd.DataFrame({
                    'Date': df_temp.iloc[:, 0],
                    'RIC': ric,                 
                    'PriceClose': df_temp.iloc[:, 1]
                })
                
                df_clean['Date'] = pd.to_datetime(df_clean['Date'])
                new_data_list.append(df_clean)
                
        except Exception as e:
            # Bỏ qua lỗi lặt vặt (ví dụ: mã bị hủy niêm yết)
            pass

# (Giữ nguyên BƯỚC 3 của cậu để gộp bảng và drop_duplicates)

🔄 BẮT ĐẦU CẬP NHẬT DATABASE CHO 1046 MÃ...
📅 DB cũ lưu đến ngày: 2026-02-25
🛡️ Kéo đè an toàn từ: 2026-02-20 đến 2026-02-25 (Sẽ tự động xóa trùng lặp)
🚀 Đang tải dữ liệu...


  4%|▍         | 41/1046 [00:17<07:15,  2.31it/s]


KeyboardInterrupt: 

In [ ]:
import kit1 as kt
print(dir(kt))  # In ra các thuộc tính và hàm của module kit để kiểm tra
returns = pd.read_csv("/Users/phamthanh/Documents/ESSCA | MSc Finance & Data Analyst/14. ESG Portfolio/v1.2/historical_data.csv")

returns.head()


['__builtins__', '__cached__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'calculate_capital_allocation', 'logging', 'np', 'pd', 'rd', 'scan_momentum_from_returns', 'scan_safe_momentum', 'scan_smart_mean_reversion', 'scan_zscore_from_returns']


,Date,A.N,A3M.MC,AA.N,AALB.AS,AAMI.N,AAP.N,AAT.N,ABBV.N,ABI.BR,...,XIOR.BR,XOM.N,XPRO.N,XYL.N,YELP.N,YUM.N,YUMC.N,ZBH.N,ZTS.N,ZWS.N
0,2025-01-03,0.016938,0.002237,-0.060016,-0.008711,0.001939,-0.037391,0.009977,0.009920,-0.028302,...,0.003413,0.005125,-0.003906,0.010608,0.012030,-0.000898,-0.042498,0.000096,0.004305,0.004336
1,2025-01-06,0.005454,0.000000,0.021843,0.020504,-0.003870,0.031075,-0.029635,-0.006180,0.003377,...,-0.015306,-0.001113,0.014902,-0.002902,0.012645,-0.024505,0.007622,-0.008711,0.015859,-0.004317
2,2025-01-07,0.007183,-0.006696,-0.006851,-0.001148,-0.022533,-0.037463,-0.023884,-0.003165,0.004207,...,-0.008636,0.009374,0.031685,-0.009072,-0.017732,-0.012445,-0.003115,-0.004152,-0.014527,-0.016260
3,2025-01-08,-0.002984,0.001124,-0.006623,-0.023563,-0.004769,-0.010002,-0.023265,-0.005737,-0.007122,...,-0.015679,-0.016736,0.018727,0.001123,-0.007882,-0.006301,-0.027449,-0.015417,0.009358,0.018182
4,2025-01-09,0.000000,0.008979,0.000000,-0.001766,0.000000,0.000000,0.000000,0.000000,0.004641,...,0.019469,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

print("⚙️ ĐANG KHỞI ĐỘNG MÔ HÌNH QUANT MOMENTUM PRO...")

# ==========================================
# BƯỚC 1: CHUẨN HOÁ LẠI DỮ LIỆU ĐẦU VÀO (Quan trọng từ ảnh chụp)
# ==========================================
# 1. Xử lý bảng returns
if 'Date' in returns.columns:
    returns['Date'] = pd.to_datetime(returns['Date'])
    returns = returns.set_index('Date')

# Ép kiểu toàn bộ bảng returns về float để tránh lỗi tính toán
returns = returns.apply(pd.to_numeric, errors='coerce')

# 2. Xử lý bảng df_wide (giá) nếu chưa set index
if 'Date' in df_wide.columns:
    df_wide['Date'] = pd.to_datetime(df_wide['Date'])
    df_wide = df_wide.set_index('Date')
    
df_wide = df_wide.apply(pd.to_numeric, errors='coerce')

# ==========================================
# BƯỚC 2: TÍNH TOÁN CÁC CHỈ SỐ ALPHA CỐT LÕI
# ==========================================
# Giải đấu ngắn hạn -> Sử dụng lookback 21 ngày (1 tháng giao dịch)
lookback = 21

# 1. Cumulative Return (Lợi nhuận tích lũy 21 ngày)
# Dùng công thức compound (1+r) thay vì chỉ cộng dồn để chính xác tuyệt đối
cum_returns_21d = (1 + returns).rolling(window=lookback).apply(np.prod, raw=True) - 1

# 2. Volatility (Độ biến động 21 ngày - Thường niên hoá)
volatility_21d = returns.rolling(window=lookback).std() * np.sqrt(252)

# 3. Risk-Adjusted Momentum (Sharpe Ratio ngắn hạn) *** CHÌA KHÓA TỐI ĐA HOÁ LỢI NHUẬN ***
# Chia đà tăng cho độ biến động giúp ta tìm ra mã tăng MẠNH nhưng ỔN ĐỊNH, ít rủi ro sập hầm
risk_adj_mom = cum_returns_21d.iloc[-1] / volatility_21d.iloc[-1]

# 4. Trend Filter (Bộ lọc xu hướng ngắn hạn)
# Tớ dùng EMA 20 (Exponential Moving Average) nhạy với giá gần nhất hơn SMA
ema_20 = df_wide.ewm(span=20, adjust=False).mean()

# Điều kiện: Giá hiện tại bắt buộc phải nằm trên EMA 20 (Xác nhận Uptrend mạnh)
current_price = df_wide.iloc[-1]
is_uptrend = current_price > ema_20.iloc[-1]

# ==========================================
# BƯỚC 3: TỔNG HỢP, XẾP HẠNG & CHỌN LỌC
# ==========================================
# Gom số liệu ngày cuối cùng vào một bảng
df_signals = pd.DataFrame({
    'Current_Price': current_price,
    'Return_21D(%)': cum_returns_21d.iloc[-1] * 100,
    'Volatility(%)': volatility_21d.iloc[-1] * 100,
    'Risk_Adj_Mom': risk_adj_mom,
    'Is_Uptrend': is_uptrend
})

# BỘ LỌC TÀN KHỐC:
# 1. Phải đang trong Uptrend (Giá > EMA 20)
# 2. Phải có lợi nhuận 21 ngày dương (Loại bắt đáy)
df_filtered = df_signals[(df_signals['Is_Uptrend'] == True) & 
                         (df_signals['Return_21D(%)'] > 0)].dropna()

# Sắp xếp theo Risk-Adjusted Momentum từ cao xuống thấp
df_top_picks = df_filtered.sort_values(by='Risk_Adj_Mom', ascending=False)

# Lấy Top 5 mã đỉnh nhất để bung sức mua
top_5_stocks = df_top_picks.head(5)

print("\n🚀 [KẾT QUẢ TÍN HIỆU] TOP 5 CỔ PHIẾU MOMENTUM MẠNH NHẤT HÔM NAY:")
display(top_5_stocks[['Current_Price', 'Return_21D(%)', 'Volatility(%)', 'Risk_Adj_Mom']])

# ==========================================
# BƯỚC 4: CHIẾN THUẬT ĐI LỆNH (POSITION SIZING DÀNH CHO PRO)
# ==========================================
print("\n💼 [QUẢN TRỊ VỐN] KHUYẾN NGHỊ PHÂN BỔ VÀ CẮT LỖ (Dựa trên Volatility Parity):")

# Tổng vốn giả định muốn giải ngân cho lượt này (Ví dụ: Cậu định đánh 5,000,000 EUR)
TOTAL_CAPITAL_ALLOCATED = 5000000 

# Phân bổ vốn nghịch đảo với độ biến động (Inverse Volatility Sizing)
# Mã nào giật mạnh -> Mua ít tiền lại / Mã nào đi êm -> Mua nhiều tiền hơn
inv_vol = 1 / top_5_stocks['Volatility(%)']
weights = inv_vol / inv_vol.sum()

for ric in top_5_stocks.index:
    price = top_5_stocks.loc[ric, 'Current_Price']
    vol = top_5_stocks.loc[ric, 'Volatility(%)'] / 100
    weight = weights.loc[ric]
    capital_for_stock = TOTAL_CAPITAL_ALLOCATED * weight
    shares_to_buy = int(capital_for_stock / price)
    
    # Tính ATR Cắt lỗ (Khoảng 2 lần độ lệch chuẩn ngày để không bị nhiễu)
    stop_loss_pct = max(0.05, vol / 4) # Tối thiểu cắt lỗ 5%
    sl_price = price * (1 - stop_loss_pct)
    
    print(f"🔥 {ric}:")
    print(f"   + Mua: {shares_to_buy:,} cổ phiếu (Trọng số: {weight:.1%}) quanh giá {price:.2f}")
    print(f"   + Cắt lỗ tự động (SL): {sl_price:.2f} (-{stop_loss_pct:.1%})")

⚙️ ĐANG KHỞI ĐỘNG MÔ HÌNH QUANT MOMENTUM PRO...

🚀 [KẾT QUẢ TÍN HIỆU] TOP 5 CỔ PHIẾU MOMENTUM MẠNH NHẤT HÔM NAY:


,Current_Price,Return_21D(%),Volatility(%),Risk_Adj_Mom
VIE.PA,35.36,14.247934,12.500427,1.139796
URW.PA,103.95,15.105541,14.054688,1.074769
FDX.N,386.56,26.133263,25.435154,1.027447
ATI.N,161.04,28.911372,28.331590,1.020464
LEGD.PA,154.80,20.476749,20.312828,1.008070



💼 [QUẢN TRỊ VỐN] KHUYẾN NGHỊ PHÂN BỔ VÀ CẮT LỖ (Dựa trên Volatility Parity):
🔥 VIE.PA:
   + Mua: 41,135 cổ phiếu (Trọng số: 29.1%) quanh giá 35.36
   + Cắt lỗ tự động (SL): 33.59 (-5.0%)
🔥 URW.PA:
   + Mua: 12,445 cổ phiếu (Trọng số: 25.9%) quanh giá 103.95
   + Cắt lỗ tự động (SL): 98.75 (-5.0%)
🔥 FDX.N:
   + Mua: 1,849 cổ phiếu (Trọng số: 14.3%) quanh giá 386.56
   + Cắt lỗ tự động (SL): 361.98 (-6.4%)
🔥 ATI.N:
   + Mua: 3,985 cổ phiếu (Trọng số: 12.8%) quanh giá 161.04
   + Cắt lỗ tự động (SL): 149.63 (-7.1%)
🔥 LEGD.PA:
   + Mua: 5,782 cổ phiếu (Trọng số: 17.9%) quanh giá 154.80
   + Cắt lỗ tự động (SL): 146.94 (-5.1%)


In [109]:
import pandas as pd
import numpy as np

# ==========================================
# 1. ÉP KIỂU VÀ LỌC SẠCH BẢNG RETURNS 
# ==========================================
# Chỉ lấy các cột chứa dữ liệu số (loại bỏ Date, Ticker nếu chúng vô tình lọt vào)
returns_numeric = returns.select_dtypes(include='number')

# Ép toàn bộ dữ liệu thành dạng số, nếu có chữ xen kẽ ("N/A") thì ép thành NaN để không bị lỗi
returns_numeric = returns_numeric.apply(pd.to_numeric, errors='coerce')

# ==========================================
# 2. TÍNH TOÁN MOMENTUM KÉP & Z-SCORE
# ==========================================
# Dùng bảng returns_numeric siêu sạch để tính toán
mom_20d = (1 + returns_numeric.tail(20)).prod() - 1
mom_60d = (1 + returns_numeric.tail(60)).prod() - 1

mean_20d = returns_numeric.tail(20).mean()
std_20d = returns_numeric.tail(20).std()
z_score = (mean_20d / std_20d) * np.sqrt(20)

# ==========================================
# 3. GỘP BẢNG VÀ ÁP DỤNG BỘ LỌC
# ==========================================
df_screener = pd.DataFrame({
    'Momentum_20D (%)': mom_20d * 100,
    'Momentum_60D (%)': mom_60d * 100,
    'Z_Score': z_score
})

MAX_Z_SCORE = 1.5

df_filtered = df_screener[
    (df_screener['Momentum_20D (%)'] > 0) &  
    (df_screener['Momentum_60D (%)'] > 0) &  
    (df_screener['Z_Score'] > 0) &           
    (df_screener['Z_Score'] <= MAX_Z_SCORE)  
].copy()

# Sắp xếp và xóa dòng trống
df_filtered = df_filtered.round(2).sort_values(by=['Momentum_60D (%)', 'Momentum_20D (%)'], ascending=[False, False])
df_filtered = df_filtered.dropna()

print(f"🎯 Đã lọc ra {len(df_filtered)} mã có đà tăng bền vững kép (20D & 60D):")
display(df_filtered.head(20))

🎯 Đã lọc ra 394 mã có đà tăng bền vững kép (20D & 60D):


,Momentum_20D (%),Momentum_60D (%),Z_Score
TROX.N,5.50,73.80,0.33
VAL.N,61.57,67.14,1.40
SCCO.N,10.83,56.96,0.56
NBR.N,16.94,55.63,1.02
FCX.N,7.16,55.52,0.51
DAN.N,14.08,54.98,1.39
CSTM.N,12.35,52.80,0.85
WT.N,3.34,52.55,0.40
GREG.MC,10.47,52.50,1.08
OII.N,27.95,52.17,1.36


In [110]:
import refinitiv.data as rd

# 1. TÁI XẾP HẠNG: Đưa Động lượng 20 ngày lên làm "Vua", Z-Score làm Á hậu
# Chuyển index thành cột Ticker để dễ xử lý
if 'Ticker' not in df_filtered.columns:
    df_filtered = df_filtered.reset_index().rename(columns={'index': 'Ticker'})

# Xếp hạng lại và cắt lấy 50 ứng viên sáng giá nhất để tiết kiệm thời gian gọi API
df_top50 = df_filtered.sort_values(by=['Momentum_20D (%)', 'Z_Score'], ascending=[False, False]).head(50).copy()

print("🔄 Đang quét Sóng Ngành và Khoảng cách SMA50 cho Top 50 ứng viên...")

top50_tickers = df_top50['Ticker'].tolist()

# 2. GỌI API LẤY NGÀNH (SECTOR)
df_sector = rd.get_data(
    universe=top50_tickers,
    fields=['TR.GICSSector']
)
df_sector = df_sector.rename(columns={'Instrument': 'Ticker', 'GICS Sector Name': 'Sector', 'TR.GICSSector': 'Sector'})

# 3. KIỂM TRA ĐỘ CĂNG SMA50
# Lấy giá của 50 mã này từ bảng df_wide (hoặc bảng giá đóng cửa bạn đang có)
# Giả sử bảng giá diện rộng chứa toàn bộ lịch sử của bạn tên là df_wide
price_slice = df_wide[top50_tickers]
latest_price = price_slice.ffill().iloc[-1]
sma50 = price_slice.rolling(window=50).mean().ffill().iloc[-1]
dist_from_sma50 = ((latest_price - sma50) / sma50) * 100

df_extension = pd.DataFrame({
    'Ticker': dist_from_sma50.index,
    'Dist_from_SMA50 (%)': dist_from_sma50.values
})

# 4. TỔNG HỢP VÀ GẠN LỌC CUỐI CÙNG
df_final = df_top50.merge(df_sector, on='Ticker', how='left').merge(df_extension, on='Ticker', how='left')

# BỘ LỌC THÉP: Cách SMA50 không quá 15%
df_final = df_final[df_final['Dist_from_SMA50 (%)'] <= 15]

# ĐA DẠNG HÓA NGÀNH: Lấy đúng 2 mã tốt nhất của mỗi ngành
df_portfolio = df_final.groupby('Sector').head(2).reset_index(drop=True)

# Chốt chặn: Lấy đúng 15 mã đứng đầu danh sách đã được đa dạng hóa
top_15_picks = df_portfolio.head(15)

# Làm tròn cho đẹp
top_15_picks['Dist_from_SMA50 (%)'] = top_15_picks['Dist_from_SMA50 (%)'].round(2)

print("\n🚀 DANH MỤC TOP 15 MÃ HOÀN HẢO ĐỂ XUỐNG TIỀN:")
display(top_15_picks[['Ticker', 'Sector', 'Momentum_20D (%)', 'Z_Score', 'Dist_from_SMA50 (%)']])

print("\n📊 PHÂN BỔ SÓNG NGÀNH TRONG DANH MỤC:")
print(top_15_picks['Sector'].value_counts())

🔄 Đang quét Sóng Ngành và Khoảng cách SMA50 cho Top 50 ứng viên...


2026-02-25 20:22:45,177 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"



🚀 DANH MỤC TOP 15 MÃ HOÀN HẢO ĐỂ XUỐNG TIỀN:


/var/folders/d3/4j2xrjxn06v329lnxwrwmzs80000gn/T/ipykernel_28875/937690236.py:48:SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


,Ticker,Sector,Momentum_20D (%),Z_Score,Dist_from_SMA50 (%)
0,LUV.N,Industrials,21.10,0.95,10.18
1,CCO.N,Communication Services,20.20,1.19,8.67
2,USFD.N,Consumer Staples,19.77,1.26,14.28
3,DECK.N,Consumer Discretionary,16.65,0.83,12.28
4,AAP.N,Consumer Discretionary,16.47,1.20,13.33
5,AKE.PA,Materials,15.80,1.07,9.96
6,ATR.N,Materials,15.33,1.39,11.88
7,R.N,Industrials,15.31,1.36,11.45
8,MCK.N,Health Care,14.84,0.84,11.09
9,PRG.N,Financials,14.52,0.91,13.24



📊 PHÂN BỔ SÓNG NGÀNH TRONG DANH MỤC:
Sector
Industrials               2
Consumer Staples          2
Consumer Discretionary    2
Materials                 2
Health Care               2
Communication Services    1
Financials                1
Real Estate               1
Energy                    1
Information Technology    1
Name: count, dtype: Int64


In [ ]:
print(df)

NameError: name 'df' is not defined

In [ ]:
# Lấy phần tử đầu tiên trong tuple gán vào một biến mới
real_df = df[0]

# In ra 10 dòng đầu tiên của DataFrame
print(real_df.head(20))
type(real_df)

NameError: name 'df' is not defined

In [ ]:
kt.calculate_capital_allocation(returns, real_df['Ticker'].head(10))

,Ticker,Độ giật rủi ro (Std Dev),Tỷ trọng vốn (%),Tiền giải ngân (VNĐ)
0,GLW.N,0.0494,5.09,4580061.0
1,VRT.N,0.0636,3.95,3558750.0
2,FDX.N,0.0163,15.46,13915727.0
3,VZ.N,0.0287,8.77,7890856.0
4,DE.N,0.0297,8.45,7606397.0
5,JCI.N,0.0178,14.14,12725575.0
6,T.N,0.0204,12.31,11079337.0
7,SGEF.PA,0.0223,11.27,10143715.0
8,HWM.N,0.0237,10.62,9558825.0
9,CAT.N,0.0253,9.93,8940758.0


In [ ]:
# ==========================================
# BƯỚC THỰC THI CHIẾN LƯỢC: SAFE MOMENTUM PRO
# ==========================================
print("🚀 ĐANG QUÉT TÌM CHIẾN MÃ MOMENTUM AN TOÀN (Lookback=20, Z-Ceiling=2.0)...")

# 1. Chạy hàm quét tín hiệu trên bảng tỷ suất sinh lời (returns)
top_safe_momentum, all_scanned_data = scan_safe_momentum(returns, lookback=20, z_ceiling=2.0)

# 2. Đánh giá và ra quyết định
if top_safe_momentum.empty:
    print("⚠️ BÁO ĐỘNG: Không có mã nào thỏa mãn tiêu chí Momentum + Z-Score an toàn hôm nay.")
    print("Khuyến nghị của Quant Manager: GIỮ TIỀN MẶT, ĐỨNG NGOÀI THỊ TRƯỜNG!")
else:
    print(f"\n🎯 TÌM THẤY {len(top_safe_momentum)} MÃ ĐẠT CHUẨN. TOP 5 MẠNH NHẤT:")
    # Chỉ lấy Top 5 mã mạnh nhất để đảm bảo tính cô đặc của danh mục
    top_5_picks = top_safe_momentum.head(5)
    display(top_5_picks)
    
    # ==========================================
    # BƯỚC PHÂN BỔ VỐN (RISK PARITY ALLOCATION)
    # ==========================================
    # Lấy danh sách mã (tickers) của top 5 mã này
    target_tickers = top_5_picks['Ticker'].tolist()
    
    # Tớ giả định quỹ vốn giải ngân đợt này của cậu là 5,000,000 EUR. 
    # Cậu có thể thay đổi con số này cho khớp với số dư thực tế trong cuộc thi nhé.
    TOTAL_CAPITAL = 5_000_000 
    
    print(f"\n💼 ĐANG TÍNH TOÁN PHÂN BỔ VỐN ({TOTAL_CAPITAL:,} EUR) THEO RISK PARITY:")
    print("Nguyên tắc: Mã nào giật mạnh (Rủi ro cao) -> Cấp ít tiền. Mã nào êm -> Cấp nhiều tiền.")
    
    df_allocation = calculate_capital_allocation(
        df_returns=returns, 
        target_tickers=target_tickers, 
        total_capital=TOTAL_CAPITAL, 
        lookback=20
    )
    
    # Sắp xếp lại bảng chia tiền cho đẹp
    df_allocation = df_allocation.sort_values(by='Tỷ trọng vốn (%)', ascending=False).reset_index(drop=True)
    display(df_allocation)
    
    print("\n✅ LỆNH GIẢI NGÂN ĐÃ SẴN SÀNG. CẬU CÓ THỂ VÀO APP ĐỂ ĐẶT LỆNH THEO CỘT 'Tiền giải ngân' CỦA BẢNG TRÊN.")

🚀 ĐANG QUÉT TÌM CHIẾN MÃ MOMENTUM AN TOÀN (Lookback=20, Z-Ceiling=2.0)...


NameError: name 'scan_safe_momentum' is not defined

In [ ]:
# 1. BỘ LỌC AN TOÀN: Chỉ giữ lại những mã có Dist_from_SMA50 <= 15% (Không mua đuổi)
df_safe = df_final_analysis[df_final_analysis['Dist_from_SMA50 (%)'] <= 15].copy()

# 2. XẾP HẠNG CHẤT LƯỢNG: Ưu tiên Z-Score (Tăng trưởng mượt), sau đó mới xét đến Momentum thuần
df_ranked = df_safe.sort_values(by=['Z-Score', 'Momentum (20D %)'], ascending=[False, False])

# 3. ĐA DẠNG HÓA NGÀNH (Sector Limit): Chọn tối đa 3 mã tốt nhất cho mỗi nhóm ngành 
# Điều này giúp tránh rủi ro tập trung (Concentration Risk)
df_diversified = df_ranked.groupby('Sector').head(3).reset_index(drop=True)

# 4. CHỐT DANH SÁCH: Cắt lấy đúng 15 mã đứng đầu danh sách đã được đa dạng hóa
top_15_portfolio = df_diversified.head(20).copy()

# Hiển thị danh mục đầu tư cuối cùng
print("🚀 DANH MỤC TOP 15 CỔ PHIẾU TỐI ƯU ĐỂ VÀO LỆNH (LONG POSITION):")
display(top_15_portfolio[['Ticker', 'Sector', 'Momentum (20D %)', 'Z-Score', 'Dist_from_SMA50 (%)']])

# Kiểm tra xem danh mục phân bổ vào các ngành như thế nào
print("\n📊 TỶ TRỌNG NGÀNH TRONG DANH MỤC 15 MÃ:")
print(top_15_portfolio['Sector'].value_counts())

🚀 DANH MỤC TOP 15 CỔ PHIẾU TỐI ƯU ĐỂ VÀO LỆNH (LONG POSITION):


🚀 DANH MỤC TOP 15 CỔ PHIẾU TỐI ƯU ĐỂ VÀO LỆNH (LONG POSITION):


,Ticker,Sector,Momentum (20D %),Z-Score,Dist_from_SMA50 (%)
0,ABT.N,Health Care,5.99,1.82,-11.082009
1,PG.N,Consumer Staples,10.49,1.77,10.01989
2,MCD.N,Consumer Discretionary,6.91,1.74,10.279212
3,DIOR.PA,Consumer Discretionary,-2.94,1.74,-8.671857
4,GE.N,Industrials,14.89,1.72,12.363461
5,SO.N,Utilities,7.96,1.62,1.508009
6,MO.N,Consumer Staples,9.53,1.61,11.0993
7,AIRP.PA,Materials,11.06,1.52,5.076829
8,TRV.N,Financials,8.41,1.46,10.040061
9,O.N,Real Estate,10.40,1.43,13.871176



📊 TỶ TRỌNG NGÀNH TRONG DANH MỤC 15 MÃ:
Sector
Health Care               3
Consumer Discretionary    3
Utilities                 3
Consumer Staples          2
Materials                 2
Financials                2
Real Estate               2
Industrials               1
Information Technology    1
Energy                    1
Name: count, dtype: Int64


In [ ]:
import pandas as pd
import refinitiv.data as rd

# 1. Dữ liệu 9 mã đã được chắt lọc
tickers = ['PG.N', 'MCD.N', 'GE.N', 'SO.N', 'MO.N', 'AXAF.PA', 'TRV.N', 'O.N', 'MSI.N']
z_scores = [1.77, 1.74, 1.72, 1.62, 1.61, 1.3, 1.46, 1.43, 1.42]

df_allocation = pd.DataFrame({'Ticker': tickers, 'Z-Score': z_scores})

# 2. Tính toán tỷ trọng vốn
total_capital = 60000000  # 60 triệu EUR
sum_z_score = df_allocation['Z-Score'].sum()

df_allocation['Weight (%)'] = (df_allocation['Z-Score'] / sum_z_score) * 100
df_allocation['Allocated_EUR'] = (df_allocation['Z-Score'] / sum_z_score) * total_capital

# 3. Kéo giá hiện tại (Quy đổi tất cả sang EUR) để tính số lượng cổ phiếu
print("🔄 Đang lấy giá khớp lệnh hiện tại...")
df_prices = rd.get_data(
    universe=tickers,
    fields=['TR.PriceClose']
)

# Ghép giá vào bảng phân bổ
df_allocation = df_allocation.merge(df_prices, left_on='Ticker', right_on='Instrument')
df_allocation = df_allocation.rename(columns={'Price Close': 'Price_EUR'})

# 4. Tính số lượng cổ phiếu cần mua (Làm tròn xuống phần nguyên)
df_allocation['Shares_to_Buy'] = (df_allocation['Allocated_EUR'] / df_allocation['Price_EUR']).astype(int)

# Định dạng hiển thị tiền
df_allocation['Allocated_EUR'] = df_allocation['Allocated_EUR'].apply(lambda x: f"€{x:,.0f}")
df_allocation['Weight (%)'] = df_allocation['Weight (%)'].apply(lambda x: f"{x:.2f}%")
df_allocation['Price_EUR'] = df_allocation['Price_EUR'].round(2)

print("\n📝 DANH SÁCH LỆNH MUA THỰC TẾ (TRADE LIST):")
display(df_allocation[['Ticker', 'Weight (%)', 'Allocated_EUR', 'Price_EUR', 'Shares_to_Buy']])

🔄 Đang lấy giá khớp lệnh hiện tại...


🔄 Đang lấy giá khớp lệnh hiện tại...


2026-02-24 10:57:02,662 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"



📝 DANH SÁCH LỆNH MUA THỰC TẾ (TRADE LIST):


,Ticker,Weight (%),Allocated_EUR,Price_EUR,Shares_to_Buy
0,PG.N,12.58%,"€7,547,974",165.17,45698
1,MCD.N,12.37%,"€7,420,043",334.56,22178
2,GE.N,12.22%,"€7,334,755",338.99,21637
3,SO.N,11.51%,"€6,908,316",95.18,72581
4,MO.N,11.44%,"€6,865,672",68.98,99531
5,AXAF.PA,9.24%,"€5,543,710",39.82,139219
6,TRV.N,10.38%,"€6,226,013",305.39,20387
7,O.N,10.16%,"€6,098,081",66.68,91452
8,MSI.N,10.09%,"€6,055,437",465.03,13021


In [ ]:
import pandas as pd
import refinitiv.data as rd
from datetime import datetime, timedelta

# 1. Đã sửa mã DG.PA thành SGEF.PA (Mã RIC chuẩn của Vinci SA)
portfolio_rics = ['BDX.N', 'PVH.N', 'SGEF.PA', 'APAM.AS', 'USFD.N']
results = []

# 2. Tính toán ngày Start và End (Lùi 100 ngày để đảm bảo luôn có dư > 70 ngày giao dịch)
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=100)).strftime('%Y-%m-%d')

print(f"🔄 Đang kéo dữ liệu an toàn từ {start_date} đến {end_date}...")

for ric in portfolio_rics:
    try:
        # Gọi API với khoảng thời gian cố định
        df_hist = rd.get_history(
            universe=ric,
            fields=['TR.PriceClose', 'M'],
            start=start_date,
            end=end_date
        )
        
        # Kiểm tra xem có dữ liệu trả về không
        if df_hist is not None and not df_hist.empty:
            # Lấy cột đầu tiên (chứa giá đóng cửa)
            df_price = df_hist.iloc[:, 0]
            
            # BỘ LỌC AN TOÀN: Phải có trên 21 ngày mới tính toán được
            if len(df_price) >= 21:
                latest_price = df_price.ffill().iloc[-1]
                price_20d_ago = df_price.ffill().iloc[-21]
                
                # Cố gắng tính SMA50 nếu có đủ 50 ngày, nếu không thì lấy trung bình toàn bộ
                sma50 = df_price.rolling(window=50).mean().ffill().iloc[-1] if len(df_price) >= 50 else df_price.mean()
                
                # Tính toán Động lượng và SMA
                momentum_20d = ((latest_price - price_20d_ago) / price_20d_ago) * 100
                dist_sma50 = ((latest_price - sma50) / sma50) * 100
                
                # Logic Hành động
                if momentum_20d < 0:
                    action = "❌ BÁN (Cắt lỗ/Cơ cấu)"
                elif dist_sma50 > 15:
                    action = "⚠️ BÁN (Chốt lời từng phần)"
                elif dist_sma50 < 0:
                    action = "❌ BÁN (Gãy trend SMA50)"
                else:
                    action = "✅ GIỮ (Hold)"
                    
                # Ghi nhận kết quả
                results.append({
                    'Mã (RIC)': ric,
                    'Giá hiện tại': latest_price,
                    'SMA 50': sma50,
                    'Momentum 20D (%)': momentum_20d,
                    'Cách SMA50 (%)': dist_sma50,
                    'Hành động': action
                })
            else:
                print(f"⚠️ Bỏ qua {ric}: Không đủ 21 ngày giao dịch để tính Momentum (chỉ có {len(df_price)} dòng).")
        else:
            print(f"⚠️ Bỏ qua {ric}: API không trả về dữ liệu.")
            
    except Exception as e:
        print(f"⚠️ Lỗi hệ thống khi xử lý {ric}: {e}")

# 3. Tạo bảng tổng hợp
if results:
    df_portfolio = pd.DataFrame(results)
    
    # Làm tròn số
    for col in ['Giá hiện tại', 'SMA 50', 'Momentum 20D (%)', 'Cách SMA50 (%)']:
        df_portfolio[col] = df_portfolio[col].round(2)
        
    # Sắp xếp từ tốt nhất đến tệ nhất
    df_portfolio = df_portfolio.sort_values(by='Momentum 20D (%)', ascending=False).reset_index(drop=True)
    
    print("\n📊 ĐÁNH GIÁ TRẠNG THÁI 5 MÃ ĐANG CẦM (REAL-TIME):")
    display(df_portfolio)
else:
    print("Không có dữ liệu nào được tính toán thành công.")

🔄 Đang kéo dữ liệu an toàn từ 2025-11-16 đến 2026-02-24...


🔄 Đang kéo dữ liệu an toàn từ 2025-11-16 đến 2026-02-24...


2026-02-24 10:05:22,788 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:23,420 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:23,420 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,033 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,033 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,582 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,582 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:25,143 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:25,143 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"


🔄 Đang kéo dữ liệu an toàn từ 2025-11-16 đến 2026-02-24...


2026-02-24 10:05:22,788 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:23,420 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:23,420 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,033 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,033 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,582 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:24,582 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:25,143 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"
2026-02-24 10:05:25,143 - HTTP Request: POST http://localhost:9005/api/udf "HTTP/1.1 200 OK"



📊 ĐÁNH GIÁ TRẠNG THÁI 5 MÃ ĐANG CẦM (REAL-TIME):


,Mã (RIC),Giá hiện tại,SMA 50,Momentum 20D (%),Cách SMA50 (%),Hành động
0,APAM.AS,43.50,37.16,22.47,17.05,⚠️ BÁN (Chốt lời từng phần)
1,SGEF.PA,140.90,123.67,20.27,13.93,✅ GIỮ (Hold)
2,USFD.N,96.13,83.02,18.28,15.79,⚠️ BÁN (Chốt lời từng phần)
3,BDX.N,184.34,162.18,16.25,13.66,✅ GIỮ (Hold)
4,PVH.N,69.12,67.42,12.67,2.52,✅ GIỮ (Hold)


In [ ]:
CFG["history_start"] = "2025-01-01"

In [ ]:
kt.scan_smart_mean_reversion(returns)

'Lỗi: Dữ liệu của bạn không đủ 200 phiên giao dịch để tính SMA 200.'